In [34]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/integrated_data.csv', parse_dates=['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

trend_cols = ['BTC_google_trend', 'ETH_google_trend', 'BNB_google_trend', 
              'ADA_google_trend', 'SOL_google_trend', 'DOGE_google_trend']

symbol_to_trend = {
    'BTC': 'BTC_google_trend', 'ETH': 'ETH_google_trend', 'BNB': 'BNB_google_trend',
    'ADA': 'ADA_google_trend', 'SOL': 'SOL_google_trend', 'DOGE': 'DOGE_google_trend'
}

def compute_features(group, symbol):
    # Work with a copy to avoid modifying original
    group = group.copy()
    
    # Price features
    group['ret_1d'] = group['close'].pct_change()
    group['ret_7d'] = group['close'].pct_change(7)
    group['ret_14d'] = group['close'].pct_change(14)
    group['ret_30d'] = group['close'].pct_change(30)
    group['log_ret'] = np.log(group['close'] / group['close'].shift(1).replace(0, np.nan))
    group['ma_7'] = group['close'].rolling(7).mean()
    group['ma_14'] = group['close'].rolling(14).mean()
    group['ma_30'] = group['close'].rolling(30).mean()
    group['ma_60'] = group['close'].rolling(60).mean()
    group['ma_ratio_7_30'] = group['ma_7'] / group['ma_30'].replace(0, np.nan)
    group['ma_ratio_30_60'] = group['ma_30'] / group['ma_60'].replace(0, np.nan)
    group['zscore_30'] = (group['close'] - group['ma_30']) / group['close'].rolling(30).std().replace(0, np.nan)
    group['zscore_60'] = (group['close'] - group['ma_60']) / group['close'].rolling(60).std().replace(0, np.nan)
    
    # Volatility
    group['daily_range'] = (group['high'] - group['low']) / group['close'].replace(0, np.nan)
    group['atr_7'] = group['daily_range'].rolling(7).mean()
    group['atr_14'] = group['daily_range'].rolling(14).mean()
    group['atr_30'] = group['daily_range'].rolling(30).mean()
    group['volatility_trend'] = group['atr_7'] / group['atr_30'].replace(0, np.nan)
    group['realized_vol_7'] = group['log_ret'].rolling(7).std() * np.sqrt(365)
    group['realized_vol_30'] = group['log_ret'].rolling(30).std() * np.sqrt(365)
    
    # Volume
    group['volume_ma_7'] = group['volume'].rolling(7).mean()
    group['volume_ma_14'] = group['volume'].rolling(14).mean()
    group['volume_ma_30'] = group['volume'].rolling(30).mean()
    group['volume_ratio_7'] = group['volume'] / group['volume_ma_7'].replace(0, np.nan)
    group['volume_ratio_30'] = group['volume'] / group['volume_ma_30'].replace(0, np.nan)
    group['volume_change_7d'] = group['volume'].pct_change(7)
    group['volume_std_30'] = group['volume'].rolling(30).std()
    group['volume_zscore'] = (group['volume'] - group['volume_ma_30']) / group['volume_std_30'].replace(0, np.nan)
    group['price_volume_corr_30'] = group['close'].rolling(30).corr(group['volume'])
    
    # RSI
    delta = group['close'].diff()
    gain = delta.clip(lower=0)
    loss = delta.clip(upper=0).abs()
    group['rsi_7'] = 100 - 100 / (1 + gain.rolling(7).mean() / loss.rolling(7).mean().replace(0, np.nan))
    group['rsi_14'] = 100 - 100 / (1 + gain.rolling(14).mean() / loss.rolling(14).mean().replace(0, np.nan))
    group['rsi_30'] = 100 - 100 / (1 + gain.rolling(30).mean() / loss.rolling(30).mean().replace(0, np.nan))
    
    # Momentum
    group['roc_7'] = (group['close'] - group['close'].shift(7)) / group['close'].shift(7).replace(0, np.nan) * 100
    group['roc_14'] = (group['close'] - group['close'].shift(14)) / group['close'].shift(14).replace(0, np.nan) * 100
    group['roc_30'] = (group['close'] - group['close'].shift(30)) / group['close'].shift(30).replace(0, np.nan) * 100
    
    # Price position with safe division
    range_30 = group['close'].rolling(30).max() - group['close'].rolling(30).min()
    range_60 = group['close'].rolling(60).max() - group['close'].rolling(60).min()
    group['price_position_30'] = (group['close'] - group['close'].rolling(30).min()) / range_30.replace(0, np.nan)
    group['price_position_60'] = (group['close'] - group['close'].rolling(60).min()) / range_60.replace(0, np.nan)
    
    # Google Trends
    if symbol in symbol_to_trend:
        trend = group[symbol_to_trend[symbol]]
        group['own_trend'] = trend
        group['own_trend_ma_7'] = trend.rolling(7).mean()
        group['own_trend_ma_14'] = trend.rolling(14).mean()
        group['own_trend_ma_30'] = trend.rolling(30).mean()
        group['own_trend_std_30'] = trend.rolling(30).std()
        group['own_trend_zscore'] = (trend - group['own_trend_ma_30']) / group['own_trend_std_30'].replace(0, np.nan)
        group['own_trend_change_1d'] = trend.diff()
        group['own_trend_change_7d'] = trend.diff(7)
        group['own_trend_momentum'] = group['own_trend_ma_7'] - group['own_trend_ma_30']
        group['trend_price_corr_30'] = trend.rolling(30).corr(group['close'])
        group['trend_volume_corr_30'] = trend.rolling(30).corr(group['volume'])
        group['trend_price_divergence'] = group['own_trend_zscore'] - group['zscore_30']
    
    group['btc_trend_ma_30'] = group['BTC_google_trend'].rolling(30).mean()
    group['btc_trend_zscore'] = (group['BTC_google_trend'] - group['btc_trend_ma_30']) / group['BTC_google_trend'].rolling(30).std().replace(0, np.nan)
    
    # Sentiment
    group['fear_greed'] = group['value']
    group['fear_greed_classification'] = group['value_classification']
    group['fear_greed_ma_7'] = group['value'].rolling(7).mean()
    group['fear_greed_ma_14'] = group['value'].rolling(14).mean()
    group['fear_greed_ma_30'] = group['value'].rolling(30).mean()
    group['fear_greed_std_30'] = group['value'].rolling(30).std()
    group['fear_greed_zscore'] = (group['value'] - group['fear_greed_ma_30']) / group['fear_greed_std_30'].replace(0, np.nan)
    group['fear_greed_change_1d'] = group['value'].diff()
    group['fear_greed_change_7d'] = group['value'].diff(7)
    group['fear_greed_change_14d'] = group['value'].diff(14)
    group['fear_greed_momentum'] = group['fear_greed_ma_7'] - group['fear_greed_ma_30']
    group['extreme_greed'] = (group['value'] > 75).astype(int)
    group['extreme_fear'] = (group['value'] < 25).astype(int)
    group['greed_streak'] = (group['value'] > 60).astype(int).groupby((group['value'] <= 60).cumsum()).cumsum()
    group['fear_streak'] = (group['value'] < 40).astype(int).groupby((group['value'] >= 40).cumsum()).cumsum()
    group['sentiment_price_corr_30'] = group['value'].rolling(30).corr(group['ret_1d'])
    group['sentiment_price_divergence'] = group['fear_greed_ma_7'] - group['zscore_30'] * 25 + 50
    
    # Macro
    group['fed_funds_change_1d'] = group['fed_funds'].diff()
    group['fed_funds_change_7d'] = group['fed_funds'].diff(7)
    group['fed_funds_change_30d'] = group['fed_funds'].diff(30)
    group['fed_funds_ma_30'] = group['fed_funds'].rolling(30).mean()
    group['treasury_10y_change_1d'] = group['treasury_10y'].diff()
    group['treasury_10y_change_7d'] = group['treasury_10y'].diff(7)
    group['treasury_10y_change_30d'] = group['treasury_10y'].diff(30)
    group['treasury_10y_ma_30'] = group['treasury_10y'].rolling(30).mean()
    group['yield_spread'] = group['treasury_10y'] - group['fed_funds']
    group['yield_spread_change'] = group['yield_spread'].diff(7)
    group['vix_ma_7'] = group['vix'].rolling(7).mean()
    group['vix_ma_30'] = group['vix'].rolling(30).mean()
    group['vix_std_30'] = group['vix'].rolling(30).std()
    group['vix_zscore'] = (group['vix'] - group['vix_ma_30']) / group['vix_std_30'].replace(0, np.nan)
    group['vix_change_1d'] = group['vix'].diff()
    group['vix_change_7d'] = group['vix'].diff(7)
    group['vix_regime'] = (group['vix'] > 20).astype(int) + (group['vix'] > 30).astype(int)
    group['m2_growth_30d'] = group['m2'].pct_change(30)
    group['m2_growth_60d'] = group['m2'].pct_change(60)
    group['m2_growth_90d'] = group['m2'].pct_change(90)
    group['m2_ma_30'] = group['m2'].rolling(30).mean()
    group['m2_acceleration'] = group['m2_growth_30d'] - group['m2_growth_30d'].shift(30)
    group['cpi_change_30d'] = group['cpi'].diff(30)
    group['cpi_ma_30'] = group['cpi'].rolling(30).mean()
    group['real_rate_change'] = group['real_rate_10y'].diff(30)
    group['price_vix_corr_30'] = group['close'].rolling(30).corr(group['vix'])
    group['price_m2_corr_60'] = group['close'].rolling(60).corr(group['m2'])
    group['risk_off_signal'] = ((group['vix'] > group['vix_ma_30']) & (group['treasury_10y'] < group['treasury_10y_ma_30'])).astype(int)
    
    # Lagged features
    for lag in [1, 3, 7, 14]:
        for col in ['ret_1d', 'ret_7d', 'volume_ratio_30', 'fear_greed', 'rsi_14']:
            group[f'{col}_lag_{lag}'] = group[col].shift(lag)
    
    # Time features
    group['day_of_week'] = group['date'].dt.dayofweek
    group['month'] = group['date'].dt.month
    group['quarter'] = group['date'].dt.quarter
    group['year'] = group['date'].dt.year
    group['is_month_start'] = group['date'].dt.is_month_start.astype(int)
    group['is_month_end'] = group['date'].dt.is_month_end.astype(int)
    
    # Drop warmup period
    group = group.iloc[90:]
    
    return group

print("Computing features per coin...")
result_list = []

for symbol, group in df.groupby('symbol'):
    print(f"  Processing {symbol}...")
    processed = compute_features(group, symbol)
    result_list.append(processed)

feature_df = pd.concat(result_list, ignore_index=True)

# NA handling with median imputation per coin
num_cols = feature_df.select_dtypes(include=[np.number]).columns
feature_df[num_cols] = feature_df.groupby('symbol')[num_cols].transform(lambda x: x.fillna(x.median()))

feature_df.to_csv('data/df_features.csv', index=False)

print(f"\nShape: {feature_df.shape}")
print(f"Coins: {feature_df['symbol'].unique().tolist()}")
print(f"Date range: {feature_df['date'].min()} to {feature_df['date'].max()}")
print(f"Rows per coin: {feature_df.groupby('symbol').size().to_dict()}")
print(f"NaNs remaining: {feature_df.select_dtypes(include=[np.number]).isnull().sum().sum()}")

Computing features per coin...
  Processing ADA...
  Processing BNB...
  Processing BTC...
  Processing DOGE...
  Processing ETH...
  Processing SOL...

Shape: (12825, 143)
Coins: ['ADA', 'BNB', 'BTC', 'DOGE', 'ETH', 'SOL']
Date range: 2018-07-30 00:00:00 to 2025-01-01 00:00:00
Rows per coin: {'ADA': 2348, 'BNB': 2348, 'BTC': 2348, 'DOGE': 1918, 'ETH': 2348, 'SOL': 1515}
NaNs remaining: 128250


In [35]:
feature_df

,date,open,high,low,close,volume,symbol,BTC_google_trend,ETH_google_trend,BNB_google_trend,...,ret_7d_lag_14,volume_ratio_30_lag_14,fear_greed_lag_14,rsi_14_lag_14,day_of_week,month,quarter,year,is_month_start,is_month_end
0,2018-07-30,0.16274,0.16300,0.14716,0.15327,1.178316e+08,ADA,7.0,2.0,12.0,...,0.123430,1.313515,36.0,51.407206,0,7,3,2018,0,0
1,2018-07-31,0.15323,0.15349,0.13900,0.14175,1.392801e+08,ADA,7.0,2.0,12.0,...,0.337287,1.798850,39.0,62.987790,1,7,3,2018,0,1
2,2018-08-01,0.14152,0.14474,0.13500,0.13984,1.255888e+08,ADA,7.0,2.0,11.0,...,0.373478,3.232114,42.0,66.115068,2,8,3,2018,1,0
3,2018-08-02,0.13983,0.14149,0.12850,0.13039,1.317920e+08,ADA,7.0,2.0,12.0,...,0.426745,2.088660,44.0,69.438506,3,8,3,2018,0,0
4,2018-08-03,0.13079,0.13473,0.12290,0.12947,1.387924e+08,ADA,7.0,2.0,10.0,...,0.171485,1.807242,47.0,57.597536,4,8,3,2018,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12820,2024-12-28,184.22000,196.10000,183.82000,195.50000,1.494848e+06,SOL,39.0,9.0,13.0,...,-0.076468,0.443926,83.0,38.876999,5,12,4,2024,0,0
12821,2024-12-29,195.50000,197.71000,188.57000,189.94000,1.560190e+06,SOL,39.0,9.0,14.0,...,-0.053793,0.448498,80.0,42.449078,6,12,4,2024,0,0
12822,2024-12-30,189.94000,196.47000,185.89000,191.38000,2.415985e+06,SOL,50.0,10.0,14.0,...,-0.002031,0.815093,83.0,44.041516,0,12,4,2024,0,0
12823,2024-12-31,191.37000,199.27000,188.04000,189.31000,2.461978e+06,SOL,43.0,9.0,14.0,...,0.044710,1.117983,87.0,43.160168,1,12,4,2024,0,1
